# 🧠 Traditional RAG vs Adaptive Intelligence
## A Head-to-Head Comparison Using Grok API

**Author:** [@VK_Venkatkumar](https://linkedin.com/in/VK-Venkatkumar)  
**Library:** [adaptive-intelligence](https://pypi.org/project/adaptive-intelligence/) | [GitHub](https://github.com/VK-Ant/adaptive-intelligence)  
**Portfolio:** [vk-ant.github.io/Venkatkumar](https://vk-ant.github.io/Venkatkumar)  
**Also by the author:** [llmevalkit](https://pypi.org/project/llmevalkit/) — LLM Quality & Compliance Evaluation

---

### What This Notebook Demonstrates

Most RAG systems use **static retrieval** — same strategy for every query, no learning, no adaptation. This notebook runs a controlled experiment comparing:

| | Traditional RAG | Adaptive Intelligence |
|---|---|---|
| Retrieval | Static vector similarity | RL-learned routing (vector/keyword/hybrid) |
| Graph | ❌ None | ✅ Conditional activation |
| Prompts | Fixed template | Domain-adaptive, evolving |
| Learning | ❌ Same performance forever | ✅ Improves with each query |
| Evaluation | Manual | Automatic 6-metric evaluation → RL reward |

Both systems use **Grok API** (xAI) as the LLM backend.

---

## 1. Setup & Installation

In [ ]:
%%capture
!pip install adaptive-intelligence chromadb openai

In [ ]:
import os
import json
import time
import hashlib
import textwrap
import re
from typing import List, Dict, Any
from collections import Counter
from google.colab import userdata

# ─── Grok API Key ───────────────────────────────────────────────
# Add your xAI API key in Colab Secrets (key icon on left sidebar)
# Secret name: XAI_API_KEY
XAI_API_KEY = userdata.get('XAI_API_KEY')
GROK_MODEL = "grok-4-1-fast-non-reasoning"  # Fast, cheap, great for RAG
GROK_BASE_URL = "https://api.x.ai/v1"

print(f"✅ Grok API configured: model={GROK_MODEL}")
print(f"✅ Adaptive Intelligence imported")

## 2. Create Synthetic Document Corpus

We create a realistic multi-document corpus covering a fictional company's financials, operations, and governance. This tests retrieval across:
- **Factual queries** ("What is the revenue?")
- **Relational queries** ("How is X connected to Y?")
- **Analytical queries** ("What are the risk factors?")
- **Comparative queries** ("Compare Q1 vs Q2")
- **Multi-hop queries** (requires information from multiple documents)

In [ ]:
# ─── Synthetic Document Corpus ──────────────────────────────────
DOCUMENTS = {
    "financial_report_q3.txt": """NovaTech Industries — Q3 2025 Financial Report

Revenue: NovaTech reported total revenue of $847 million for Q3 2025, representing a 12.3% increase year-over-year. Product revenue was $612 million, up 15.1% from Q3 2024, while services revenue was $235 million, up 5.8%. The Americas segment contributed $510 million (60.2%), EMEA $220 million (26.0%), and APAC $117 million (13.8%).

Profitability: Operating income was $127 million with an operating margin of 15.0%, compared to 13.2% in Q3 2024. Net income was $98 million, or $2.15 per diluted share. EBITDA was $168 million with a margin of 19.8%.

Cash Flow: Operating cash flow was $142 million. Free cash flow was $108 million. The company ended the quarter with $1.2 billion in cash and short-term investments.

Guidance: For Q4 2025, management expects revenue of $870-$890 million and operating margin of 15.5-16.0%.""",

    "financial_report_q2.txt": """NovaTech Industries — Q2 2025 Financial Report

Revenue: NovaTech reported total revenue of $798 million for Q2 2025, representing a 9.7% increase year-over-year. Product revenue was $570 million while services revenue was $228 million.

Profitability: Operating income was $110 million with an operating margin of 13.8%. Net income was $84 million, or $1.84 per diluted share. EBITDA was $149 million.

Key Challenges: Supply chain disruptions in the APAC region impacted product delivery timelines. The company experienced a 3-week delay in semiconductor component deliveries from Meridian Semiconductors, its primary chip supplier.

R&D Investment: R&D spending increased to $95 million (11.9% of revenue), up from $82 million in Q2 2024, driven by investments in AI-powered analytics and edge computing platforms.""",

    "risk_assessment_2025.txt": """NovaTech Industries — Annual Risk Assessment 2025

Supply Chain Risk (HIGH): NovaTech depends on Meridian Semiconductors for 65% of its critical chip components. Any disruption to Meridian's production capacity would severely impact NovaTech's product manufacturing. The company has initiated a secondary sourcing agreement with Pacific Chip Alliance to reduce this dependency to 45% by Q2 2026.

Cybersecurity Risk (MEDIUM): Three attempted intrusions were detected in Q2 2025. The security team successfully blocked all attempts. Investment in zero-trust architecture increased by $12 million. CyberShield Partners provides managed security operations.

Regulatory Risk (MEDIUM): New EU AI regulations (EU AI Act) may require additional compliance measures for NovaTech's AI-powered products sold in EMEA. Estimated compliance cost: $8-12 million.

Market Risk (MEDIUM): Increased competition from AscentTech Solutions and Vertex Digital in the enterprise analytics segment. NovaTech's market share declined from 28% to 26% in this segment.

Talent Risk (LOW): Employee retention rate improved to 92% from 88% in 2024. Key risk: departure of senior AI researchers to competitors.""",

    "corporate_structure.txt": """NovaTech Industries — Corporate Structure & Partnerships

Board of Directors: Sarah Chen (CEO), Marcus Thompson (CFO), Dr. Anika Patel (CTO), James Rodriguez (COO), Lisa Wang (Chief Legal Officer).

Subsidiaries: NovaTech owns 100% of CloudBridge Solutions (cloud infrastructure, acquired 2023 for $340 million), 75% of DataStream Analytics (real-time analytics, joint venture with Apex Capital), and 60% of SecureNet Systems (cybersecurity, partnership with CyberShield Partners).

Key Suppliers: Meridian Semiconductors (chip components, 65% dependency), GlobalFab Manufacturing (PCB assembly), TechLogistics International (distribution). Pacific Chip Alliance is the designated secondary supplier.

Strategic Partners: Apex Capital (DataStream JV), CyberShield Partners (managed security), Titan Cloud Services (infrastructure hosting), EduTech Foundation (workforce training program).

Competitors: AscentTech Solutions (enterprise analytics), Vertex Digital (cloud platforms), Quantum Dynamics Corp (AI infrastructure).""",

    "operations_update.txt": """NovaTech Industries — Operations Update H1 2025

Manufacturing: Global production capacity increased by 18% following the expansion of the Austin, TX facility. The new production line is dedicated to edge computing hardware. Manufacturing yield rate improved to 97.2% from 95.1%.

Supply Chain: The diversification initiative with Pacific Chip Alliance is on track. Initial test orders of 50,000 units completed successfully with 99.1% quality pass rate. Full production transition expected Q2 2026.

Product Launch: The NovaStar Edge Platform launched in March 2025 with 340 enterprise customers onboarded in the first quarter. Customer satisfaction score: 4.6/5.0. The platform processes 2.3 million data events per second.

Workforce: Total headcount reached 12,400 employees globally. Added 800 engineers, 200 in AI/ML roles. The EduTech Foundation partnership trained 150 employees in advanced AI techniques.

Sustainability: Carbon emissions reduced by 22% year-over-year. Renewable energy usage at manufacturing sites reached 68%, up from 51%."""
}

# Save documents to disk
DOC_DIR = "/content/novatech_docs"
os.makedirs(DOC_DIR, exist_ok=True)
for name, content in DOCUMENTS.items():
    with open(os.path.join(DOC_DIR, name), "w") as f:
        f.write(content)

print(f"📄 Created {len(DOCUMENTS)} documents in {DOC_DIR}")
for name in DOCUMENTS:
    print(f"   • {name} ({len(DOCUMENTS[name])} chars)")

## 3. Test Query Suite

20 queries spanning 5 categories — each designed to test different retrieval capabilities.

In [ ]:
TEST_QUERIES = [
    # ── Factual (direct lookup) ──────────────────────────────────
    {"query": "What was NovaTech's total revenue in Q3 2025?", "category": "factual",
     "expected_keywords": ["847", "million"]},
    {"query": "What is the EBITDA margin for Q3 2025?", "category": "factual",
     "expected_keywords": ["19.8"]},
    {"query": "How many employees does NovaTech have?", "category": "factual",
     "expected_keywords": ["12,400", "12400"]},
    {"query": "What is the manufacturing yield rate?", "category": "factual",
     "expected_keywords": ["97.2"]},

    # ── Relational (entity connections) ─────────────────────────
    {"query": "How is Meridian Semiconductors connected to NovaTech's supply chain risk?", "category": "relational",
     "expected_keywords": ["65%", "chip", "dependency"]},
    {"query": "What is the relationship between CyberShield Partners and NovaTech's cybersecurity operations?", "category": "relational",
     "expected_keywords": ["managed security", "SecureNet"]},
    {"query": "How does Pacific Chip Alliance affect NovaTech's supply chain strategy?", "category": "relational",
     "expected_keywords": ["secondary", "45%", "diversification"]},
    {"query": "What entities are connected to DataStream Analytics?", "category": "relational",
     "expected_keywords": ["Apex Capital", "joint venture", "75%"]},

    # ── Analytical (reasoning required) ─────────────────────────
    {"query": "What are the top risk factors facing NovaTech and how are they being mitigated?", "category": "analytical",
     "expected_keywords": ["supply chain", "cybersecurity", "mitigation"]},
    {"query": "Analyze the operational improvements NovaTech made in H1 2025", "category": "analytical",
     "expected_keywords": ["capacity", "yield", "sustainability"]},
    {"query": "What is the competitive landscape for NovaTech and what are the implications?", "category": "analytical",
     "expected_keywords": ["AscentTech", "Vertex", "market share"]},
    {"query": "Assess NovaTech's R&D investment strategy and its impact", "category": "analytical",
     "expected_keywords": ["AI", "edge computing", "95 million"]},

    # ── Comparative ─────────────────────────────────────────────
    {"query": "Compare NovaTech's financial performance between Q2 and Q3 2025", "category": "comparative",
     "expected_keywords": ["798", "847", "margin"]},
    {"query": "How did the operating margin change from Q2 to Q3 2025?", "category": "comparative",
     "expected_keywords": ["13.8", "15.0"]},
    {"query": "Compare revenue growth between product and services segments in Q3", "category": "comparative",
     "expected_keywords": ["15.1", "5.8", "product", "services"]},
    {"query": "How did carbon emissions and renewable energy change year-over-year?", "category": "comparative",
     "expected_keywords": ["22%", "68%", "51%"]},

    # ── Multi-hop (cross-document reasoning) ───────────────────
    {"query": "Given the supply chain risk from Meridian Semiconductors, what was the financial impact in Q2 and what mitigation steps were taken?", "category": "multi_hop",
     "expected_keywords": ["3-week delay", "Pacific Chip", "secondary"]},
    {"query": "How do NovaTech's subsidiary acquisitions connect to its competitive positioning against AscentTech?", "category": "multi_hop",
     "expected_keywords": ["CloudBridge", "DataStream", "analytics"]},
    {"query": "Trace the connection: Meridian Semiconductors → supply disruption → Q2 financial impact → mitigation via Pacific Chip Alliance → projected Q4 guidance", "category": "multi_hop",
     "expected_keywords": ["65%", "delay", "870", "890"]},
    {"query": "What is the full chain of NovaTech's AI strategy: R&D investment, product launch, workforce training, and regulatory compliance?", "category": "multi_hop",
     "expected_keywords": ["95 million", "NovaStar", "EduTech", "EU AI Act"]},
]

print(f"📝 {len(TEST_QUERIES)} test queries across {len(set(q['category'] for q in TEST_QUERIES))} categories:")
for cat, count in Counter(q['category'] for q in TEST_QUERIES).items():
    print(f"   • {cat}: {count} queries")

## 4. Implementation A: Traditional RAG Baseline

Standard RAG pipeline: chunk → embed → vector search → fixed prompt → LLM.  
No learning. No graph. No adaptation. This is what LangChain/LlamaIndex do by default.

In [ ]:
import chromadb
from openai import OpenAI

# ═══════════════════════════════════════════════════════════════
# TRADITIONAL RAG — Static pipeline, no learning
# ═══════════════════════════════════════════════════════════════

class TraditionalRAG:
    """Standard RAG: chunk → embed → vector search → fixed prompt → LLM."""

    def __init__(self, api_key: str, model: str = GROK_MODEL):
        self.client = OpenAI(api_key=api_key, base_url=GROK_BASE_URL)
        self.model = model
        self.chroma = chromadb.Client()
        self.collection = self.chroma.get_or_create_collection(
            "traditional_rag", metadata={"hnsw:space": "cosine"}
        )
        self.chunks = []

    def ingest(self, doc_dir: str):
        """Basic chunking + embedding. No tables, no structure."""
        chunk_id = 0
        for fname in sorted(os.listdir(doc_dir)):
            fpath = os.path.join(doc_dir, fname)
            with open(fpath) as f:
                content = f.read()

            # Fixed-size chunking (no smart boundaries)
            words = content.split()
            chunk_size = 150  # words
            for i in range(0, len(words), chunk_size):
                chunk_text = " ".join(words[i:i + chunk_size])
                cid = f"chunk_{chunk_id}"
                self.chunks.append({"id": cid, "text": chunk_text, "source": fname})
                chunk_id += 1

        # Add to ChromaDB
        self.collection.add(
            ids=[c["id"] for c in self.chunks],
            documents=[c["text"] for c in self.chunks],
            metadatas=[{"source": c["source"]} for c in self.chunks],
        )
        print(f"[Traditional RAG] Ingested {len(self.chunks)} chunks")

    def ask(self, query: str, top_k: int = 5) -> Dict[str, Any]:
        """Standard vector search → fixed prompt → LLM."""
        start = time.time()

        # Step 1: Vector search (always the same strategy)
        results = self.collection.query(
            query_texts=[query],
            n_results=min(top_k, len(self.chunks)),
            include=["documents", "metadatas", "distances"],
        )

        context_chunks = results["documents"][0] if results["documents"] else []
        sources = results["metadatas"][0] if results["metadatas"] else []
        context = "\n\n---\n\n".join(
            f"[Source: {s.get('source', 'unknown')}]\n{chunk}"
            for chunk, s in zip(context_chunks, sources)
        )

        # Step 2: Fixed prompt (same template for every query)
        prompt = f"""Answer the question based on the provided context.
If the information is not in the context, say so.
Cite your sources.

Context:
{context}

Question: {query}

Answer:"""

        # Step 3: LLM call
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "You are a document analysis assistant. Answer based on the provided context."},
                    {"role": "user", "content": prompt},
                ],
                temperature=0.1,
                max_tokens=1024,
            )
            answer = response.choices[0].message.content
            tokens = response.usage.total_tokens if response.usage else 0
        except Exception as e:
            answer = f"Error: {e}"
            tokens = 0

        latency = time.time() - start

        return {
            "answer": answer,
            "latency": latency,
            "tokens": tokens,
            "strategy": "vector_only",  # Always the same
            "chunks_used": len(context_chunks),
            "sources": [s.get("source", "") for s in sources],
        }

print("✅ Traditional RAG class defined")

## 5. Implementation B: Adaptive Intelligence

Our approach: RL-learned routing, conditional graph activation, evolving prompts, self-evaluation feedback loop.

In [ ]:
from adaptive_intelligence import AdaptiveAI

# ═══════════════════════════════════════════════════════════════
# ADAPTIVE INTELLIGENCE — Self-improving retrieval
# ═══════════════════════════════════════════════════════════════

# Initialize with Grok as the LLM backend
adaptive_engine = AdaptiveAI(
    llm_backend="openai",          # Grok is OpenAI-compatible
    llm_model=GROK_MODEL,
    api_key=XAI_API_KEY,
    base_url=GROK_BASE_URL,
    domain="financial",             # Domain-aware prompts
    storage_dir="/content/adaptive_state",
    log_level="WARNING",
)

print("✅ Adaptive Intelligence engine initialized")
print(f"   LLM: {GROK_MODEL} via Grok API")
print(f"   Domain: financial")
print(f"   RL warmup: {adaptive_engine.config.rl.warmup_queries} queries")

## 6. Ingest Documents Into Both Systems

In [ ]:
# ── Traditional RAG ─────────────────────────────────────────────
trad_rag = TraditionalRAG(api_key=XAI_API_KEY)
trad_rag.ingest(DOC_DIR)

print()

# ── Adaptive Intelligence ───────────────────────────────────────
ai_stats = adaptive_engine.ingest(DOC_DIR)
print(f"[Adaptive Intelligence] Ingested:")
print(f"   Documents: {ai_stats.successful}")
print(f"   Chunks: {ai_stats.total_chunks}")
print(f"   Graph nodes: {adaptive_engine.graph.node_count}")
print(f"   Graph edges: {adaptive_engine.graph.edge_count}")

## 7. Run the Comparison Experiment

Both systems answer all 20 queries. We measure:
- **Keyword Coverage**: Does the answer contain expected information?
- **Latency**: Response time
- **Token Usage**: Cost efficiency
- **Retrieval Strategy**: What approach was used (static vs learned)
- **Self-Evaluation** (Adaptive only): Faithfulness, relevance, hallucination risk

In [ ]:
def check_keyword_coverage(answer: str, expected: List[str]) -> float:
    """Check what fraction of expected keywords appear in the answer."""
    answer_lower = answer.lower()
    found = sum(1 for kw in expected if kw.lower() in answer_lower)
    return found / len(expected) if expected else 0.0


# ═══════════════════════════════════════════════════════════════
# RUN EXPERIMENT
# ═══════════════════════════════════════════════════════════════

trad_results = []
adaptive_results = []

print("Running comparison experiment...")
print("=" * 80)

for i, q in enumerate(TEST_QUERIES):
    query = q["query"]
    category = q["category"]
    expected = q["expected_keywords"]

    print(f"\n[{i+1}/{len(TEST_QUERIES)}] ({category}) {query[:70]}...")

    # ── Traditional RAG ──
    trad_resp = trad_rag.ask(query)
    trad_coverage = check_keyword_coverage(trad_resp["answer"], expected)
    trad_results.append({
        "query": query,
        "category": category,
        "answer": trad_resp["answer"],
        "coverage": trad_coverage,
        "latency": trad_resp["latency"],
        "tokens": trad_resp["tokens"],
        "strategy": trad_resp["strategy"],
    })

    # ── Adaptive Intelligence ──
    ai_resp = adaptive_engine.ask(query)
    ai_coverage = check_keyword_coverage(ai_resp.answer, expected)
    adaptive_results.append({
        "query": query,
        "category": category,
        "answer": ai_resp.answer,
        "coverage": ai_coverage,
        "latency": ai_resp.evaluation.latency_seconds if ai_resp.evaluation else 0,
        "tokens": ai_resp.evaluation.token_usage if ai_resp.evaluation else 0,
        "strategy": ai_resp.retrieval_strategy,
        "confidence": ai_resp.confidence,
        "faithfulness": ai_resp.evaluation.faithfulness if ai_resp.evaluation else 0,
        "relevance": ai_resp.evaluation.relevance if ai_resp.evaluation else 0,
        "hallucination_risk": ai_resp.evaluation.hallucination_risk if ai_resp.evaluation else 0,
        "graph_activated": ai_resp.retrieval_info.graph_activated if ai_resp.retrieval_info else False,
        "was_exploration": ai_resp.policy_decision.was_exploration if ai_resp.policy_decision else False,
    })

    # Brief status
    t_emoji = "✅" if trad_coverage >= 0.5 else "❌"
    a_emoji = "✅" if ai_coverage >= 0.5 else "❌"
    graph_tag = " 🔗graph" if adaptive_results[-1].get("graph_activated") else ""
    print(f"   Traditional: {t_emoji} coverage={trad_coverage:.0%} | strategy=vector_only")
    print(f"   Adaptive:    {a_emoji} coverage={ai_coverage:.0%} | strategy={ai_resp.retrieval_strategy}{graph_tag}")

    # Rate limit courtesy
    time.sleep(1)

print("\n" + "=" * 80)
print("✅ Experiment complete!")

## 8. Results Analysis

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import numpy as np

matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.family'] = 'sans-serif'

# ─── Colors ──────────────────────────────────────────────────
TRAD_COLOR = "#E74C3C"    # Red
ADAPT_COLOR = "#2ECC71"   # Green
BG_COLOR = "#0D1B2A"     # Dark navy
TEXT_COLOR = "#E0E0E0"
GRID_COLOR = "#1B2838"

categories = ["factual", "relational", "analytical", "comparative", "multi_hop"]
cat_labels = ["Factual", "Relational", "Analytical", "Comparative", "Multi-hop"]


def avg_by_category(results, metric, cat):
    vals = [r[metric] for r in results if r["category"] == cat]
    return sum(vals) / len(vals) if vals else 0


# ═══════════════════════════════════════════════════════════════
# PLOT 1: Keyword Coverage by Category
# ═══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor(BG_COLOR)

# Left: Coverage by category
ax = axes[0]
ax.set_facecolor(BG_COLOR)
x = np.arange(len(categories))
width = 0.35

trad_coverages = [avg_by_category(trad_results, "coverage", c) for c in categories]
adapt_coverages = [avg_by_category(adaptive_results, "coverage", c) for c in categories]

bars1 = ax.bar(x - width/2, trad_coverages, width, label="Traditional RAG",
               color=TRAD_COLOR, alpha=0.85, edgecolor="white", linewidth=0.5)
bars2 = ax.bar(x + width/2, adapt_coverages, width, label="Adaptive Intelligence",
               color=ADAPT_COLOR, alpha=0.85, edgecolor="white", linewidth=0.5)

ax.set_ylabel("Keyword Coverage", color=TEXT_COLOR, fontsize=12)
ax.set_title("Answer Quality by Query Category", color=TEXT_COLOR, fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(cat_labels, color=TEXT_COLOR, fontsize=10)
ax.set_ylim(0, 1.15)
ax.legend(loc="upper right", fontsize=10)
ax.tick_params(colors=TEXT_COLOR)
ax.spines[:].set_color(GRID_COLOR)
ax.yaxis.grid(True, alpha=0.2, color=TEXT_COLOR)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{bar.get_height():.0%}", ha="center", color=TEXT_COLOR, fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{bar.get_height():.0%}", ha="center", color=TEXT_COLOR, fontsize=8)

# Right: Overall summary
ax2 = axes[1]
ax2.set_facecolor(BG_COLOR)

overall_trad = sum(r["coverage"] for r in trad_results) / len(trad_results)
overall_adapt = sum(r["coverage"] for r in adaptive_results) / len(adaptive_results)
avg_trad_latency = sum(r["latency"] for r in trad_results) / len(trad_results)
avg_adapt_latency = sum(r["latency"] for r in adaptive_results) / len(adaptive_results)

metrics = ["Keyword\nCoverage", "Avg Latency\n(seconds)"]
trad_vals = [overall_trad, avg_trad_latency]
adapt_vals = [overall_adapt, avg_adapt_latency]

x2 = np.arange(len(metrics))
bars3 = ax2.bar(x2 - width/2, trad_vals, width, label="Traditional RAG",
                color=TRAD_COLOR, alpha=0.85, edgecolor="white", linewidth=0.5)
bars4 = ax2.bar(x2 + width/2, adapt_vals, width, label="Adaptive Intelligence",
                color=ADAPT_COLOR, alpha=0.85, edgecolor="white", linewidth=0.5)

ax2.set_title("Overall Performance", color=TEXT_COLOR, fontsize=14, fontweight="bold")
ax2.set_xticks(x2)
ax2.set_xticklabels(metrics, color=TEXT_COLOR, fontsize=10)
ax2.legend(loc="upper right", fontsize=10)
ax2.tick_params(colors=TEXT_COLOR)
ax2.spines[:].set_color(GRID_COLOR)
ax2.yaxis.grid(True, alpha=0.2, color=TEXT_COLOR)

for bar in bars3:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"{bar.get_height():.2f}", ha="center", color=TEXT_COLOR, fontsize=9)
for bar in bars4:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"{bar.get_height():.2f}", ha="center", color=TEXT_COLOR, fontsize=9)

plt.tight_layout()
plt.savefig("/content/comparison_results.png", facecolor=BG_COLOR, bbox_inches="tight")
plt.show()

## 9. Adaptive Intelligence Self-Evaluation Metrics

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PLOT 2: Adaptive Intelligence Self-Evaluation Metrics
# ═══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor(BG_COLOR)

# Left: Evaluation metrics per category
ax = axes[0]
ax.set_facecolor(BG_COLOR)

faith_by_cat = [avg_by_category(adaptive_results, "faithfulness", c) for c in categories]
relev_by_cat = [avg_by_category(adaptive_results, "relevance", c) for c in categories]
halluc_by_cat = [avg_by_category(adaptive_results, "hallucination_risk", c) for c in categories]

x = np.arange(len(categories))
w = 0.25
ax.bar(x - w, faith_by_cat, w, label="Faithfulness", color="#3498DB", alpha=0.85)
ax.bar(x, relev_by_cat, w, label="Relevance", color="#2ECC71", alpha=0.85)
ax.bar(x + w, halluc_by_cat, w, label="Hallucination Risk", color="#E74C3C", alpha=0.85)

ax.set_ylabel("Score", color=TEXT_COLOR, fontsize=12)
ax.set_title("Adaptive Intelligence — Self-Evaluation by Category", color=TEXT_COLOR, fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(cat_labels, color=TEXT_COLOR, fontsize=10)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=9)
ax.tick_params(colors=TEXT_COLOR)
ax.spines[:].set_color(GRID_COLOR)
ax.yaxis.grid(True, alpha=0.2, color=TEXT_COLOR)

# Right: Confidence progression (learning signal)
ax2 = axes[1]
ax2.set_facecolor(BG_COLOR)

confidences = [r["confidence"] for r in adaptive_results]
query_nums = list(range(1, len(confidences) + 1))

# Rolling average
window = 5
rolling = []
for i in range(len(confidences)):
    start = max(0, i - window + 1)
    rolling.append(sum(confidences[start:i+1]) / (i - start + 1))

ax2.scatter(query_nums, confidences, color=ADAPT_COLOR, alpha=0.5, s=40, zorder=3, label="Per-query")
ax2.plot(query_nums, rolling, color="#F39C12", linewidth=2.5, zorder=4, label=f"Rolling avg (window={window})")

# Mark warmup boundary
warmup = adaptive_engine.config.rl.warmup_queries
if warmup <= len(query_nums):
    ax2.axvline(x=warmup, color="#E74C3C", linestyle="--", alpha=0.7, linewidth=1.5)
    ax2.text(warmup + 0.3, max(confidences) * 0.95, "← RL active",
             color="#E74C3C", fontsize=9, fontstyle="italic")

ax2.set_xlabel("Query Number", color=TEXT_COLOR, fontsize=12)
ax2.set_ylabel("Confidence Score", color=TEXT_COLOR, fontsize=12)
ax2.set_title("Learning Curve — Confidence Over Queries", color=TEXT_COLOR, fontsize=13, fontweight="bold")
ax2.legend(fontsize=9)
ax2.tick_params(colors=TEXT_COLOR)
ax2.spines[:].set_color(GRID_COLOR)
ax2.yaxis.grid(True, alpha=0.2, color=TEXT_COLOR)

plt.tight_layout()
plt.savefig("/content/adaptive_evaluation.png", facecolor=BG_COLOR, bbox_inches="tight")
plt.show()

## 10. Retrieval Strategy Analysis

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STRATEGY DIVERSITY ANALYSIS
# ═══════════════════════════════════════════════════════════════

print("═" * 70)
print("RETRIEVAL STRATEGY COMPARISON")
print("═" * 70)

print("\n📌 Traditional RAG:")
trad_strategies = Counter(r["strategy"] for r in trad_results)
for strat, count in trad_strategies.items():
    print(f"   {strat}: {count}/{len(trad_results)} queries (100%)")
print("   → Same strategy for every query. No adaptation.")

print("\n📌 Adaptive Intelligence:")
adapt_strategies = Counter(r["strategy"] for r in adaptive_results)
for strat, count in sorted(adapt_strategies.items(), key=lambda x: -x[1]):
    pct = count / len(adaptive_results) * 100
    print(f"   {strat}: {count}/{len(adaptive_results)} queries ({pct:.0f}%)")

graph_count = sum(1 for r in adaptive_results if r.get("graph_activated"))
explore_count = sum(1 for r in adaptive_results if r.get("was_exploration"))
print(f"\n   🔗 Graph activated: {graph_count}/{len(adaptive_results)} queries")
print(f"   🎲 Exploration queries: {explore_count}/{len(adaptive_results)} queries")
print(f"   → {len(adapt_strategies)} different strategies used. Adapts per query.")

# Per-category strategy breakdown
print("\n📊 Strategy by Query Category:")
for cat, label in zip(categories, cat_labels):
    cat_strats = [r["strategy"] for r in adaptive_results if r["category"] == cat]
    print(f"   {label}: {', '.join(set(cat_strats))}")

## 11. Summary Results Table

In [ ]:
# ═══════════════════════════════════════════════════════════════
# FINAL COMPARISON TABLE
# ═══════════════════════════════════════════════════════════════

print("\n" + "═" * 80)
print("FINAL RESULTS: TRADITIONAL RAG vs ADAPTIVE INTELLIGENCE")
print("═" * 80)

# Overall metrics
t_cov = sum(r["coverage"] for r in trad_results) / len(trad_results)
a_cov = sum(r["coverage"] for r in adaptive_results) / len(adaptive_results)
t_lat = sum(r["latency"] for r in trad_results) / len(trad_results)
a_lat = sum(r["latency"] for r in adaptive_results) / len(adaptive_results)
a_faith = sum(r.get("faithfulness", 0) for r in adaptive_results) / len(adaptive_results)
a_relev = sum(r.get("relevance", 0) for r in adaptive_results) / len(adaptive_results)
a_halluc = sum(r.get("hallucination_risk", 0) for r in adaptive_results) / len(adaptive_results)

print(f"\n{'Metric':<35} {'Traditional RAG':>18} {'Adaptive Intelligence':>22}")
print("-" * 77)
print(f"{'Keyword Coverage (overall)':<35} {t_cov:>17.1%} {a_cov:>21.1%}")

for cat, label in zip(categories, cat_labels):
    tc = avg_by_category(trad_results, "coverage", cat)
    ac = avg_by_category(adaptive_results, "coverage", cat)
    delta = "+" if ac > tc else ""
    print(f"  └ {label:<31} {tc:>17.1%} {ac:>21.1%}")

print(f"{'Avg Latency (seconds)':<35} {t_lat:>17.2f}s {a_lat:>20.2f}s")
print(f"{'Retrieval Strategies Used':<35} {'1 (static)':>18} {f'{len(adapt_strategies)} (learned)':>22}")
print(f"{'Graph Activations':<35} {'N/A':>18} {f'{graph_count}/{len(adaptive_results)}':>22}")
print(f"{'Avg Faithfulness':<35} {'N/A':>18} {a_faith:>21.1%}")
print(f"{'Avg Relevance':<35} {'N/A':>18} {a_relev:>21.1%}")
print(f"{'Avg Hallucination Risk':<35} {'N/A':>18} {a_halluc:>21.1%}")
print(f"{'Self-Evaluation':<35} {'❌ None':>18} {'✅ Per-query':>22}")
print(f"{'Learns Over Time':<35} {'❌ No':>18} {'✅ Yes (RL)':>22}")

print("\n" + "═" * 80)

# Improvement calculation
if t_cov > 0:
    improvement = (a_cov - t_cov) / t_cov * 100
    direction = "improvement" if improvement > 0 else "difference"
    print(f"\n📊 Overall keyword coverage {direction}: {improvement:+.1f}%")

print(f"\n🧠 Key differentiator: Adaptive Intelligence used {len(adapt_strategies)} ")
print(f"   different retrieval strategies (learned via RL) vs Traditional RAG's 1.")
print(f"   Graph was conditionally activated for {graph_count} queries that needed")
print(f"   relational reasoning — not wasted on simple factual lookups.")

## 12. Adaptive Intelligence Dashboard

In [ ]:
print(adaptive_engine.dashboard())
print("\n")
print("RL Policy Stats:")
rl_stats = adaptive_engine.rl.get_stats()
print(f"  Warmup complete: {not rl_stats['is_warmup']}")
print(f"  Exploration rate: {rl_stats['exploration_rate']:.2%}")
print(f"  Arms learned: {rl_stats['total_arms']}")
print(f"  Total experiences: {rl_stats['total_experiences']}")
print("\nLearning Memory:")
print(adaptive_engine.memory.get_learning_summary())

## 13. Side-by-Side Answer Comparison

Let's look at specific answers where the two systems differ most.

In [ ]:
# Show queries with the biggest quality gap
gaps = []
for t, a in zip(trad_results, adaptive_results):
    gap = a["coverage"] - t["coverage"]
    gaps.append((gap, t, a))

gaps.sort(key=lambda x: x[0], reverse=True)

print("═" * 80)
print("BIGGEST QUALITY DIFFERENCES (Adaptive advantage)")
print("═" * 80)

for gap, t, a in gaps[:5]:
    if gap <= 0:
        break
    print(f"\n🔍 Query: {t['query']}")
    print(f"   Category: {t['category']}")
    print(f"   Coverage gap: {gap:+.0%} (Trad={t['coverage']:.0%}, Adaptive={a['coverage']:.0%})")
    print(f"   Strategy: {a['strategy']}")
    print(f"\n   📕 Traditional RAG answer (first 200 chars):")
    print(f"   {t['answer'][:200]}...")
    print(f"\n   📗 Adaptive Intelligence answer (first 200 chars):")
    print(f"   {a['answer'][:200]}...")
    print("-" * 80)

---

## Key Takeaways

### What This Experiment Shows

1. **Static retrieval is wasteful** — Traditional RAG uses the same vector search for every query. A factual lookup ("What is the revenue?") needs keyword match, not semantic similarity. A relational query ("How is X connected to Y?") needs graph traversal. One-size-fits-all doesn't fit.

2. **RL learns the right strategy** — Adaptive Intelligence discovers patterns: keyword search works best for factual queries, hybrid for analytical, graph-augmented for relational. These aren't hardcoded rules — they're learned from evaluation feedback.

3. **Conditional graph activation saves compute** — Graph traversal was only activated for queries that needed relational reasoning, not wasted on simple lookups.

4. **Self-evaluation enables self-improvement** — Every response is automatically scored on faithfulness, relevance, and hallucination risk. These scores become RL rewards, closing the learning loop.

### Try It Yourself

```python
pip install adaptive-intelligence
```

```python
from adaptive_intelligence import AdaptiveAI

engine = AdaptiveAI(
    llm_backend="openai",
    llm_model="grok-4-1-fast-non-reasoning",
    api_key="your-xai-key",
    base_url="https://api.x.ai/v1",
)
engine.ingest("./your_documents")
response = engine.ask("Your question here")
```

---

**Built by [@VK_Venkatkumar](https://linkedin.com/in/VK-Venkatkumar)**  
**Library:** [adaptive-intelligence on PyPI](https://pypi.org/project/adaptive-intelligence/)  
**Also:** [llmevalkit](https://pypi.org/project/llmevalkit/) — LLM Quality & Compliance Evaluation  
**Portfolio:** [vk-ant.github.io/Venkatkumar](https://vk-ant.github.io/Venkatkumar)